# 1. 패키지 설치

In [1]:
!pip install -q -U transformers datasets accelerate peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 147.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.6 MB/s eta 0:00:00


# 2. 라이브러리 불러오기

In [2]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

# 3. 모델 불러오기

In [3]:
model_id = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

model.config.use_cache = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

# 4. 이모지 설명 학습 데이터 만들기

In [4]:
train_data = [
    {
        "instruction": "😊 이 이모지는 무슨 뜻이야?",
        "output": "😊는 기분이 좋거나 상대방에게 부드럽고 친근하게 말하고 싶을 때 사용하는 이모지입니다. 카톡에서는 '고마워 😊', '좋아 😊'처럼 따뜻한 분위기를 만들 때 자주 씁니다."
    },
    {
        "instruction": "😂 이 이모지를 한국어로 설명해줘.",
        "output": "😂는 너무 웃겨서 눈물이 날 정도라는 의미입니다. 카톡에서는 친구가 웃긴 말을 했을 때 '진짜 웃기다 😂'처럼 사용합니다."
    },
    {
        "instruction": "😭 이모지의 의미를 설명해줘.",
        "output": "😭는 크게 우는 표정입니다. 실제로 슬플 때도 쓰지만, 카톡에서는 너무 감동했거나 너무 웃길 때 과장해서 쓰기도 합니다."
    },
    {
        "instruction": "👍 이 이모지는 언제 사용해?",
        "output": "👍는 좋다, 동의한다, 확인했다는 의미입니다. 카톡에서는 '알겠어 👍', '좋아 👍'처럼 짧게 긍정의 뜻을 전할 때 사용합니다."
    },
    {
        "instruction": "🙏 이모지를 설명해줘.",
        "output": "🙏는 부탁, 감사, 미안함을 표현할 때 사용합니다. 카톡에서는 '부탁해 🙏', '고마워 🙏', '미안해 🙏'처럼 예의 있게 말하고 싶을 때 자주 씁니다."
    },
    {
        "instruction": "❤️ 이모지는 무슨 뜻이야?",
        "output": "❤️는 사랑, 애정, 호감, 응원을 표현하는 이모지입니다. 가족, 친구, 연인에게 따뜻한 마음을 전할 때 사용합니다."
    },
    {
        "instruction": "🤣 이 이모지를 한국어로 설명해줘.",
        "output": "🤣는 너무 웃겨서 바닥을 구르며 웃는다는 의미입니다. 😂보다 더 강한 웃음을 표현할 때 사용합니다."
    },
    {
        "instruction": "😅 이모지의 의미는?",
        "output": "😅는 웃고 있지만 살짝 당황하거나 민망한 상황을 나타냅니다. 예를 들어 실수했을 때 '아 제가 잘못 봤네요 😅'처럼 사용할 수 있습니다."
    },
    {
        "instruction": "😢 이 이모지를 설명해줘.",
        "output": "😢는 조용히 슬프거나 속상한 마음을 표현하는 이모지입니다. 😭보다 덜 과장된 슬픔을 나타냅니다."
    },
    {
        "instruction": "😎 이모지는 어떤 뜻이야?",
        "output": "😎는 자신감, 멋짐, 여유로움을 표현하는 이모지입니다. 카톡에서는 장난스럽게 '나 좀 멋진데? 😎'처럼 사용할 수 있습니다."
    },
    {
        "instruction": "😡 이 이모지는 무슨 의미야?",
        "output": "😡는 화남, 짜증, 불만을 표현합니다. 카톡에서는 진짜 화가 났을 때도 쓰지만, 친구 사이에서는 장난스럽게 삐진 느낌으로 쓰기도 합니다."
    },
    {
        "instruction": "🥺 이모지 의미를 설명해줘.",
        "output": "🥺는 간절함, 부탁, 감동, 서운함을 표현하는 이모지입니다. 카톡에서는 '제발 부탁이야 🥺'처럼 귀엽게 부탁할 때 자주 사용합니다."
    },
    {
        "instruction": "🔥 이 이모지는 어떻게 써?",
        "output": "🔥는 멋지다, 반응이 뜨겁다, 열정적이다는 의미입니다. 예를 들어 '오늘 발표 완전 좋았어 🔥'처럼 칭찬할 때 사용할 수 있습니다."
    },
    {
        "instruction": "🎉 이모지의 의미는?",
        "output": "🎉는 축하, 파티, 기쁜 일을 표현하는 이모지입니다. 생일, 합격, 성공, 기념일 같은 상황에서 사용합니다."
    },
    {
        "instruction": "💯 이 이모지를 설명해줘.",
        "output": "💯는 완벽하다, 최고다, 100점이라는 의미입니다. 카톡에서는 '오늘 발표 💯'처럼 매우 잘했다는 칭찬으로 사용할 수 있습니다."
    }
]

# 5. Qwen 대화 형식으로 변환

In [5]:
def make_chat_text(example):
    messages = [
        {
            "role": "system",
            "content": "너는 카카오톡에서 사용하는 이모지의 의미를 한국어로 쉽게 설명하는 AI다."
        },
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["output"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}


dataset = Dataset.from_list(train_data)
dataset = dataset.map(make_chat_text)

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

# 6. LoRA 설정

In [6]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

# 7. 학습설정

In [8]:
training_args = SFTConfig(
    output_dir="./qwen3-emoji-korean-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=5,
    logging_steps=1,
    save_strategy="epoch",
    # max_seq_length=512,
    bf16=True,
    fp16=False,
    report_to="none"
)

/tmp/ipykernel_5834/1298856078.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


**현재 시스템(Colab 환경)에는 0.10.0 버전이 깔려 있는데, 최신 peft는 0.16.0 이상의 버전을 요구하고 있습니다.**

In [11]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 107.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [12]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=tokenizer
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,3.241324
2,2.641856
3,1.861059
4,1.425377
5,1.073045
6,1.049451
7,0.775584
8,0.741612
9,0.616868
10,0.554804


TrainOutput(global_step=20, training_loss=0.896140170097351, metrics={'train_runtime': 58.0635, 'train_samples_per_second': 1.292, 'train_steps_per_second': 0.344, 'total_flos': 167524307635200.0, 'train_loss': 0.896140170097351, 'epoch': 5.0})

# 9. LoRA 어댑터 저장

In [13]:
trainer.model.save_pretrained("./qwen3-emoji-korean-lora")
tokenizer.save_pretrained("./qwen3-emoji-korean-lora")

('./qwen3-emoji-korean-lora/tokenizer_config.json',
 './qwen3-emoji-korean-lora/chat_template.jinja',
 './qwen3-emoji-korean-lora/tokenizer.json')

# 10. 학습된 LoRA 모델 불러오기

In [15]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    "./qwen3-emoji-korean-lora"
)

fine_tuned_model.eval()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

# 11. 테스트 함수 만들기

In [16]:
def ask_easy(question):
    messages = [
        {
            "role": "system",
            "content": "너는 초등학생도 이해할 수 있게 아주 쉽게 설명하는 AI 선생님이다. 어려운 말은 피하고, 짧은 문장과 쉬운 비유를 사용한다."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned_model.device)

    with torch.no_grad():
        outputs = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 12. 테스트하기

In [17]:
print(ask_easy("파인튜닝을 쉽게 설명해줘."))
print(ask_easy("Transformer가 뭐야?"))
print(ask_easy("AI가 왜 틀린 답을 할 수 있어?"))

system
너는 초등학생도 이해할 수 있게 아주 쉽게 설명하는 AI 선생님이다. 어려운 말은 피하고, 짧은 문장과 쉬운 비유를 사용한다.
user
파인튜닝을 쉽게 설명해줘.
assistant
파인튜닝은 '세부 조정'이라는 의미야.
system
너는 초등학생도 이해할 수 있게 아주 쉽게 설명하는 AI 선생님이다. 어려운 말은 피하고, 짧은 문장과 쉬운 비유를 사용한다.
user
Transformer가 뭐야?
assistant
Transformer는 컴퓨터가 말을 이해하거나 글을 짧게 만들 때 쓰는 '도구'야. 🤖

이 도구는 '어떤 단어가 이전에 나왔는지'를 기억해서, 다음 말을 더 자연스럽게 만들 수 있어. 예를 들어 '나는 카카오톡에서 카톡을 했어'처럼, 이전에 카톡을 했다는 걸 기억해서 자연스럽게 말을 이어가. 💬

이 도구는 기존에 썼던 방법보다 더 빠르고 정확해. 그래서 번역, 요약, 질문에 답하는 데 아주 잘 써.
system
너는 초등학생도 이해할 수 있게 아주 쉽게 설명하는 AI 선생님이다. 어려운 말은 피하고, 짧은 문장과 쉬운 비유를 사용한다.
user
AI가 왜 틀린 답을 할 수 있어?
assistant
좋아! AI가 틀린 답을 할 수 있는 이유는, '모든 것을 아는 것'이 아니라 '학습한 것'이기 때문이야. 📚

예를 들어, 초등학생이 숫자를 더하는 법을 배웠을 때, 실수할 수 있어. AI도 마찬가지야. 학습한 예시에서 잘못 배운 경우가 있으면, 그걸 반복해서 잘못한 답을 내올 수 있어.

또한, 질문이 너무 복잡하거나 모호할 때도 틀릴 수 있어. 예를 들어, '어제는 뭐 먹었어?'라고 물어봤는데, AI는 그날 먹은 음식을 몰라서 '아침에 샌드위치 먹었어요'라고 말할 수도 있어. 😅

결국, AI는 인간처럼 완벽하게 생각하지


## 파인튜닝 모델과 일반모델 비교하기

In [18]:

model_id = "Qwen/Qwen3-4B-Instruct-2507"
adapter_path = "./qwen3-emoji-korean-lora"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# 1. 파인튜닝하지 않은 원본 모델
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)
base_model.eval()

# 2. 파인튜닝한 LoRA 모델
ft_base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

ft_model = PeftModel.from_pretrained(
    ft_base_model,
    adapter_path
)
ft_model.eval()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

## 답변 생성 함수

In [19]:
def generate_answer(model, emoji):
    messages = [
        {
            "role": "system",
            "content": "너는 카카오톡에서 사용하는 이모지의 의미를 한국어로 쉽게 설명하는 AI다."
        },
        {
            "role": "user",
            "content": f"{emoji} 이 이모지를 한국어로 설명해줘. 카톡에서는 언제 쓰는지도 예문과 함께 알려줘."
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.3,
            top_p=0.9,
            do_sample=True
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

## 테스트
- 동일 질문 비교

In [20]:
test_emojis = ["😂", "🥺", "🔥", "🙏", "😅", "💯"]

for emoji in test_emojis:
    print("=" * 80)
    print(f"테스트 이모지: {emoji}")

    print("\n[원본 모델 답변]")
    print(generate_answer(base_model, emoji))

    print("\n[파인튜닝 모델 답변]")
    print(generate_answer(ft_model, emoji))

테스트 이모지: 😂

[원본 모델 답변]
system
너는 카카오톡에서 사용하는 이모지의 의미를 한국어로 쉽게 설명하는 AI다.
user
😂 이 이모지를 한국어로 설명해줘. 카톡에서는 언제 쓰는지도 예문과 함께 알려줘.
assistant
물론이죠! 😂 이 이모지를 한국어로 쉽게 설명해드릴게요.

---

### 😂 이모지의 의미 (한국어 설명)
**"웃음" 또는 "재미있는 상황, 웃음이 나게 하는 일"을 나타냅니다.**  
사람이 웃는 모습을 표현하는 이모지로, 상황에 따라 **웃음이 나게 했거나, 웃음이 나야 할 상황**, 또는 **무섭거나 이상한 상황에서 웃음이 나게 했을 때** 쓰는 것이에요.

👉 단순히 "웃음"을 표현하는 것뿐만 아니라, **무서운 상황에서 웃음이 나게 했을 때**, **재미있는 말을 했을 때**, **무심한 태도로 상황을 받아들이는 태도**

[파인튜닝 모델 답변]
system
너는 카카오톡에서 사용하는 이모지의 의미를 한국어로 쉽게 설명하는 AI다.
user
😂 이 이모지를 한국어로 설명해줘. 카톡에서는 언제 쓰는지도 예문과 함께 알려줘.
assistant
😂는 너무 웃겨서 눈물이 날 정도라는 의미입니다. 카톡에서는 너무 웃겨서 농담처럼 말할 때 사용합니다. 예를 들어 '너가 이거 말했네 😂'처럼 친구가 말한 내용이 너무 웃겨서 반응할 때 씁니다.
테스트 이모지: 🥺

[원본 모델 답변]
system
너는 카카오톡에서 사용하는 이모지의 의미를 한국어로 쉽게 설명하는 AI다.
user
🥺 이 이모지를 한국어로 설명해줘. 카톡에서는 언제 쓰는지도 예문과 함께 알려줘.
assistant
물론이죠! 😊

---

### 🥺 이 이모지의 의미 (한국어 설명)

**🥺** 는 "눈물 흘리는 얼굴"을 나타내는 이모지로,  
**예민하거나, 슬프거나, 힘들어하거나, 조심스럽게 요청하거나, 울고 싶은 마음을 표현**할 때 쓰는 이모지예요.

쉽게 말해보면,  
👉 "아, 정말 힘들어서요",  
👉 "이거 좀 해주세요",  
👉 "아니, 좀 더 이해

# 자동 점수 평가 코드

In [ ]:
def simple_score(answer):
    score = 0

    keywords = [
        "카톡",
        "사용",
        "의미",
        "예",
        "때",
        "감정",
        "표현"
    ]

    for keyword in keywords:
        if keyword in answer:
            score += 1

    return score


for emoji in test_emojis:
    base_answer = generate_answer(base_model, emoji)
    ft_answer = generate_answer(ft_model, emoji)

    base_score = simple_score(base_answer)
    ft_score = simple_score(ft_answer)

    print("=" * 60)
    print(f"이모지: {emoji}")
    print(f"원본 모델 점수: {base_score}")
    print(f"파인튜닝 모델 점수: {ft_score}")